# BioLangFusion benchmark inputs

This notebook checks the data used to build and compare BioLangFusion-style models. It supports Phase A in [Gantt/Details task 17](../../docs/Research_Project_Plan_Gantt.xlsx): compare the paper's encoders and fusion heads on the available benchmarks, keeping departures and unresolved preprocessing explicit. A similar score is interpretable only when we understand which rows, targets and evaluation splits produced it.

The audit answers four questions, in order:

1. Which of the five reported benchmarks are available locally?
2. What does each row supply, and what biological quantity does its target measure?
3. Does one sequence-length filter reproduce the paper's reported retained counts?
4. What do repeated sequences and sequence-format checks mean for later comparisons?

No encoder is loaded and no embedding is computed here. [Step 2](blf_token_embeddings.ipynb) prepares frozen token embeddings; [step 3](blf_build_check.ipynb) checks the fusion and prediction heads. The shared [dataset registry and loader](../shared_code/datasets.py) carry the selected rows into those notebooks. See the [Stage 3 guide](README.md) for the connected investigation.


---

## Setup

Use the [notebook environment](../../README.md#notebook-dependencies) and the existing benchmark CSVs in the repository's `datasets/` folder. Reading the saved tables needs no execution; running this audit reads those local CSVs without loading encoders or downloading data.

The three cells separate imports, project paths and shared code, then the audit settings and registry summary.


In [ ]:
# Standard-library paths and module lookup.
import sys
from pathlib import Path

# Tabular summaries of the supplied CSVs.
import pandas as pd


In [ ]:
# Find Code/ from this folder or its parents; retain the repository-root fallback.
CODE_DIR = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "shared_code").is_dir()
    ),
    Path.cwd() / "Code",
)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from shared_code import datasets


In [1]:
pd.set_option("display.width", 120)

# Use the registry's benchmark order and shared length rule throughout the audit.
BENCHMARKS = datasets.BLF_BENCHMARKS
print("Registered BioLangFusion benchmarks:", ", ".join(BENCHMARKS))
print(
    "Length rule applied by blf_benchmark_rows:",
    datasets.BLF_LENGTH_LIMIT,
    "nucleotides",
)


Registered BioLangFusion benchmarks: fungal_expression, ecoli_proteins, mrna_stability, cov_vac_degradation
Length rule applied by blf_benchmark_rows: 999 nucleotides


---

## The five benchmarks BioLangFusion reports

BioLangFusion combines representations from Nucleotide Transformer v2 100M multi-species, RNA-FM and ESM-2 8M. Our implementation extracts frozen encoder features and trains downstream fusion and prediction heads. The table below contains the paper's reported scores and retained sequence counts, not results calculated by this notebook. [BioLangFusion paper](../../Papers/core/BioLangFusion.pdf).

The first cell declares those reference values. The following cell calculates the best reported single-encoder score, its difference from the reported best fusion score, and which benchmarks have a local registry entry. Those comparisons provide reference points; matching a score alone would not verify an implementation.


In [ ]:
# Published scores and retained counts, not measurements made by this audit.
# NT denotes NT v2 100M multi-species; E. coli accuracy is in percent.
PUBLISHED = pd.DataFrame(
    [
        (
            "Fungal expression", "fungal_expression", "Spearman", 3138,
            0.804, 0.767, 0.695, 0.824, "gated MIL",
        ),
        (
            "E. coli proteins", "ecoli_proteins", "accuracy", 4450,
            41.3, 52.9, 49.0, 53.9, "cross-attention",
        ),
        (
            "mRNA stability", "mrna_stability", "Spearman", 23929,
            0.530, 0.553, 0.539, 0.563, "gated MIL",
        ),
        (
            "Ab1 antibody mRNA", None, "Spearman", 723,
            0.732, 0.743, 0.711, 0.765, "cross-attention",
        ),
        (
            "CoV-Vac degradation", "cov_vac_degradation", "Spearman", 2400,
            0.780, 0.841, 0.806, 0.864, "gated MIL",
        ),
    ],
    columns=[
        "benchmark", "registry key", "metric", "used", "NT",
        "RNA-FM", "ESM-2 8M", "best fused", "best head",
    ],
)


In [2]:
PUBLISHED = PUBLISHED.assign(**{
    "best single": PUBLISHED[["NT", "RNA-FM", "ESM-2 8M"]].max(axis=1),
})
PUBLISHED = PUBLISHED.assign(**{
    "fusion gain": (PUBLISHED["best fused"] - PUBLISHED["best single"]).round(3),
    "in hand": PUBLISHED["registry key"].notna().map({True: "yes", False: "NO"}),
})
PUBLISHED.set_index("benchmark")[
    [
        "metric", "used", "in hand", "NT", "RNA-FM", "ESM-2 8M",
        "best single", "best fused", "best head", "fusion gain",
    ]
]


,metric,used,in hand,NT,RNA-FM,ESM-2 8M,best single,best fused,best head,fusion gain
benchmark,,,,,,,,,,
Fungal expression,Spearman,3138,yes,0.804,0.767,0.695,0.804,0.824,gated MIL,0.020
E. coli proteins,accuracy,4450,yes,41.300,52.900,49.000,52.900,53.900,cross-attention,1.000
mRNA stability,Spearman,23929,yes,0.530,0.553,0.539,0.553,0.563,gated MIL,0.010
Ab1 antibody mRNA,Spearman,723,NO,0.732,0.743,0.711,0.743,0.765,cross-attention,0.022
CoV-Vac degradation,Spearman,2400,yes,0.780,0.841,0.806,0.841,0.864,gated MIL,0.023


The reported gains are small: 0.010 to 0.023 in Spearman correlation across the four regression benchmarks, and 1.0 percentage point of accuracy on E. coli. This table supplies no uncertainty estimates. The later [Phase A comparison](blf_reference_benchmarks.ipynb) separates repeated training seeds and paired prediction differences; the published values here stay fixed reference values.

**Ab1 is not in hand.** Four benchmark files are available from the CodonBERT fine-tuning release recorded in the registry. BioLangFusion describes the antibody set as 723 heavy-chain sequences of up to 1,203 nucleotides, but it was not located in the inspected release. The available comparison therefore covers four of five benchmarks. Obtaining Ab1 remains separate work; no substitute dataset is introduced here.


---

## What is in each file

The four available benchmarks share one schema: `Sequence`, `Value`, `Dataset`, and the published `Split`. The audit below reports each file as distributed, before project preprocessing. The shared column name `Value` does not mean that the four datasets measure the same biological quantity.

| Benchmark | What the stored target measures | How to read a larger value |
| --- | --- | --- |
| [mRNA stability](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) | A sign-harmonized decay slope, standardized within its experimental profile; the release combines nine profiles across cell types, species and assays. | Relatively greater stability within that profile, not a specified number of hours. |
| [Fungal expression](../../docs/methods/data-and-targets.md#fungal-expression) | Natural log of the source RNA-seq expression field `Exprs`; the original field's precise abundance units remain unresolved. | Greater measured RNA abundance on the source scale, not directly greater protein yield. |
| [E. coli protein expression](../../docs/methods/data-and-targets.md#protein-expression-in-e-coli) | Categories formed from the original visual gel scores: scores 0–2 become class 0 (low), 3–4 become class 1 (medium), and 5 becomes class 2 (high). | A higher expression category; class IDs are not concentration units. |
| [CoV-Vac degradation](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna) | Mean of 68 normalized nucleotide-level degradation measurements at 50 °C with magnesium, then standardized across the released dataset. | Greater chemical degradation under that condition, not greater stability. |

These transformations are already present in the CSVs; this audit does not apply them again. The linked guide traces the measurements, transformations and worked examples. Its distinctions are needed before interpreting a distribution of `Value` or comparing predictive scores across benchmarks.


In [3]:
# Inspect every supplied row before the shared loader drops missing or long sequences.
audit_rows = []
for benchmark in BENCHMARKS:
    spec = datasets.DATASETS[benchmark]
    raw_rows = pd.read_csv(datasets.DEFAULT_DATA_DIR / spec.path)
    sequences = raw_rows[spec.sequence_column].astype(str).str.strip().str.upper()
    targets = raw_rows[spec.label_column]
    audit_rows.append({
        "benchmark": spec.label,
        "file": spec.path,
        "rows": len(raw_rows),
        "columns": ", ".join(raw_rows.columns),
        # This alphabet preview uses only the first 500 rows; it is not a full-file check.
        "alphabet": "".join(sorted(set("".join(sequences.head(500))))),
        "length min": int(sequences.str.len().min()),
        "length median": int(sequences.str.len().median()),
        "length max": int(sequences.str.len().max()),
        "task": spec.task,
        "label min": round(float(targets.min()), 3),
        "label max": round(float(targets.max()), 3),
        "distinct labels": int(targets.nunique()),
    })

AUDIT = pd.DataFrame(audit_rows).set_index("benchmark")
AUDIT


,file,rows,columns,alphabet,length min,length median,length max,task,label min,label max,distinct labels
benchmark,,,,,,,,,,,
Fungal expression,Fungal_expression.csv,7089,"Sequence, Value, Dataset, Split",ACGU,150,1086,3063,regression,-8.977,5.14,5134
"E. coli protein expression (low, medium, high)",E.Coli_proteins.csv,6348,"Sequence, Value, Dataset, Split",ACGU,171,747,3000,classification,0.000,2.00,3
mRNA stability,mRNA_Stability.csv,65356,"Sequence, Value, Dataset, Split",ACGU,3,1239,3066,regression,-3.109,2.89,65192
CoV-Vac degradation,CoV_Vaccine_Degradation.csv,2400,"Sequence, Value, Dataset, Split",ACGU,81,81,81,regression,-7.154,6.51,2378


The saved inventory shows RNA letters in the first 500 rows of each file; the alphabet column is a preview, not a check of every sequence. The later [sequence checks](#Sequence-checks) examine all retained rows for RNA letters and lengths divisible by three. Those properties describe input format; they do not verify a biological coding sequence, start position or reading frame.

Three of the four available benchmarks have continuous targets. E. coli has three class IDs, so its downstream score is classification accuracy; the regression benchmarks use Spearman correlation to assess predicted ordering.

CoV-Vac illustrates why format and biological identity must remain separate. Its fixed 81-nucleotide inputs are cropped, designed RNA fragments, not verified complete coding sequences. Each target averages degradation measurements from the original construct's first 68 positions, a different window from the 81-nucleotide input. The [target trace](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna) explains this distinction; it matters when those fragments are later translated computationally for the protein encoder.


---

## Recovering BioLangFusion's preprocessing

The available files contain more rows than BioLangFusion reports using for three benchmarks. Its discussion of input-length limits motivates checking a length filter, although the paper describes truncation rather than explicitly specifying the row filter used here.

The sweep tests every limit from 900 to 1,100 nucleotides and reports which limits match all four published retained counts. Agreement supports this reconstruction of the filter; it cannot identify the authors' exact rows or exclude other selection rules that produce the same totals.


In [4]:
published_counts = (
    PUBLISHED.dropna(subset=["registry key"])
    .set_index("registry key")["used"]
    .to_dict()
)

# Require one limit to match every benchmark; all() stops on the first mismatch.
matching_limits = [
    length_limit
    for length_limit in range(900, 1101)
    if all(
        len(datasets.blf_benchmark_rows(benchmark, length_limit=length_limit))
        == published_counts[benchmark]
        for benchmark in BENCHMARKS
    )
]
print("Length limits reproducing all four published counts:", matching_limits)

SWEEP = pd.DataFrame([
    {
        "benchmark": datasets.DATASETS[benchmark].label,
        # A deliberately loose limit disables the length cut; other loader rules still apply.
        "rows in file": len(
            datasets.blf_benchmark_rows(benchmark, length_limit=10**9)
        ),
        "kept at 999 nt": len(datasets.blf_benchmark_rows(benchmark)),
        "published used": published_counts[benchmark],
        "matches": (
            len(datasets.blf_benchmark_rows(benchmark))
            == published_counts[benchmark]
        ),
    }
    for benchmark in BENCHMARKS
]).set_index("benchmark")
SWEEP


Length limits reproducing all four published counts: [999, 1000, 1001]


,rows in file,kept at 999 nt,published used,matches
benchmark,,,,
Fungal expression,7089,3138,3138,True
"E. coli protein expression (low, medium, high)",6348,4450,4450,True
mRNA stability,65356,23929,23929,True
CoV-Vac degradation,2400,2400,2400,True


The saved sweep finds that **keeping sequences of at most 999 nucleotides and dropping longer rows** matches all four retained counts. Limits of 1,000 and 1,001 give the same counts in this sweep. We use the shared loader's 999-nucleotide cutoff; this is a length in bases, not a common tokenizer limit for all three encoders.

The [shared loader](../shared_code/datasets.py) applies that rule without truncating retained sequences, so later notebooks use the same row-selection procedure. Dropping long rows changes which examples remain; truncating would instead change their sequence inputs. This notebook adopts neither a new crop nor a new filter.

**Count agreement leaves historical uncertainty.** BioLangFusion's Appendix A.2 reports 41,123 raw mRNA-stability sequences, while the local inventory contains 65,356; both reach a used count of 23,929. The discrepancy does not establish whether the authors used different file bytes, additional filtering or a different counting convention. Matching the used total does not prove matching row membership. The [target trace](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) describes our supplied labels, while the authors' exact historical input remains unverified.

The paper's Appendix A.2 describes truncating RNA sequences exceeding the input budget, yet its reported counts decrease—for example, fungal expression from 7,056 to 3,138 and mRNA stability from 41,123 to 23,929. Truncation alone cannot reduce row counts. The implemented dropping rule matches the reported used totals, but does not resolve that difference between the prose, counts and available files. [BioLangFusion paper, Appendix A.2 and Table 2](../../Papers/core/BioLangFusion.pdf).

Ab1 remains an unresolved counterexample: the paper reports a maximum length of 1,203 nucleotides and 723 rows both before and after preprocessing. That is compatible with truncation and would not follow the same 999-nucleotide dropping rule. Its unavailable file prevents checking how it was treated. These uncertainties are retained in [Deviations](#Deviations-recorded-for-Phase-A).


---

## Published splits, and how much the test split reuses

After applying the length filter, the audit keeps each file's supplied train, validation and test assignments. It then checks whether the same normalized sequence appears in multiple partitions. This is exact sequence reuse, not merely shared codons or similar sequences.

The table distinguishes rows from distinct sequences: “test rows seen in train” counts observations, whereas “test sequences unseen” counts distinct test strings absent from both training and validation.


In [5]:
split_rows = []
for benchmark in BENCHMARKS:
    spec = datasets.DATASETS[benchmark]
    retained_rows = datasets.blf_benchmark_rows(benchmark)
    split_assignment = retained_rows[spec.split_column]
    train_sequences = set(
        retained_rows.loc[split_assignment == "train", "sequence"]
    )
    validation_sequences = set(
        retained_rows.loc[split_assignment == "val", "sequence"]
    )
    test_rows = retained_rows[split_assignment == "test"]
    test_sequences = set(test_rows["sequence"])
    repeated_targets = retained_rows.groupby("sequence")[spec.label_column]
    sequence_sizes, target_spreads = (
        repeated_targets.size(),
        repeated_targets.max() - repeated_targets.min(),
    )

    split_rows.append({
        "benchmark": spec.label,
        "train": int((split_assignment == "train").sum()),
        "val": int((split_assignment == "val").sum()),
        "test": int((split_assignment == "test").sum()),
        "distinct sequences": int(retained_rows["sequence"].nunique()),
        "sequences in >1 split": int((
            retained_rows.groupby("sequence")[spec.split_column].nunique() > 1
        ).sum()),
        # Count test observations, including repeated copies of the same sequence.
        "test rows seen in train": int(
            test_rows["sequence"].isin(train_sequences).sum()
        ),
        # Set subtraction counts distinct strings absent from both fitting partitions.
        "test sequences unseen": len(
            test_sequences - train_sequences - validation_sequences
        ),
        "repeats with differing labels": int(
            (target_spreads[sequence_sizes > 1] > 0).sum()
        ),
    })

SPLITS = pd.DataFrame(split_rows).set_index("benchmark")
SPLITS


,train,val,test,distinct sequences,sequences in >1 split,test rows seen in train,test sequences unseen,repeats with differing labels
benchmark,,,,,,,,
Fungal expression,2153,555,430,3131,0,0,430,6
"E. coli protein expression (low, medium, high)",3010,728,712,4450,0,0,712,0
mRNA stability,16742,3608,3579,11303,3178,2444,985,4549
CoV-Vac degradation,1600,400,400,2400,0,0,400,0


No exact normalized sequence from fungal expression, E. coli or CoV-Vac appears in more than one split in the saved audit. This does not establish independence of species, gene families or experimental contexts.

**The mRNA stability benchmark has exact sequence overlap.** Its 23,929 retained rows contain 11,303 distinct sequences. Of its 3,579 test rows, 2,444 (about 68%) carry a sequence that also appears in training. Only 985 distinct test sequences are absent from both training and validation.

Repeated sequences can also carry different labels. The [source trace](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) establishes that this benchmark combines nine experimental profiles across species, cell types and assays, with labels standardized within each profile. Different observations of the same sequence can therefore reflect different biological or measurement contexts; they are not automatically inconsistent replicate measurements.

The published-split comparison retains those observations and assignments. Its score mixes predictions for previously seen sequence strings and unseen strings, rather than measuring only generalization to new sequences. Familiarity may help prediction, but this audit does not measure its effect on model scores.

A sequence-grouped comparison would keep every occurrence of a sequence on one side of the train/test boundary. That is a separate follow-up, not a second split constructed by this notebook. The [Stage 4 limitations and remaining work](../stage4/README.md#proposed-tests-of-fusion-explanations) distinguish the retained published-split comparison from additional evaluation.


## Sequence checks

With the row filter and supplied splits established, the audit checks how the retained sequences can enter the codon-grid pipeline: RNA alphabet, length divisible by three, and an AUG start. It also reports the fraction that the existing Stage 1 filter would reject.

The Stage 1 [`is_retained` filter](../shared_code/datasets.py) checks the sequence format and requires a translation of at least five amino acids. Here it is evaluated as a diagnostic only; its result does not remove rows from the BioLangFusion benchmarks.


In [6]:
sequence_checks = []
for benchmark in BENCHMARKS:
    spec = datasets.DATASETS[benchmark]
    sequences = datasets.blf_benchmark_rows(benchmark)["sequence"]
    sequence_checks.append({
        "benchmark": spec.label,
        "kept rows": len(sequences),
        # Unlike the raw-file preview, this alphabet check uses every retained row.
        "RNA letters only": bool(set("".join(sequences)) <= set("ACGU")),
        # Boolean means are fractions from 0 to 1, displayed to three decimals.
        "length a multiple of 3": f"{(sequences.str.len() % 3 == 0).mean():.3f}",
        "starts with AUG": f"{(sequences.str[:3] == 'AUG').mean():.3f}",
        # Report rejection by the Stage 1 rule without applying it to this cohort.
        "dropped by the Stage 1 filters": (
            f"{1 - sequences.map(datasets.is_retained).mean():.3f}"
        ),
    })

CHECKS = pd.DataFrame(sequence_checks).set_index("benchmark")
CHECKS


,kept rows,RNA letters only,length a multiple of 3,starts with AUG,dropped by the Stage 1 filters
benchmark,,,,,
Fungal expression,3138,True,1.000,0.932,0.068
"E. coli protein expression (low, medium, high)",4450,True,1.000,1.000,0.000
mRNA stability,23929,True,1.000,0.978,0.022
CoV-Vac degradation,2400,True,1.000,0.016,0.989


The saved table explains why the Stage 1 filter is not reused here: only 1.6% of CoV-Vac fragments begin with AUG, and the full Stage 1 rule would reject 98.9% of that benchmark. The rule would also reject 6.8% of retained fungal rows and 2.2% of retained stability rows. These are diagnostic rejection rates; this notebook keeps the rows selected by the BioLangFusion length filter.

The protein-input choice is already implemented in [step 2](blf_token_embeddings.ipynb): [`grid_protein`](../shared_code/fusion/token_embeddings.py) translates complete triplets from the supplied sequence origin and replaces stop codons with `X`, retaining one protein-input position per triplet. It does not search for a start codon or the longest open reading frame.

That computational rule preserves positions for the codon grid. It does not establish a biological reading frame or a naturally produced protein for CoV-Vac's designed RNA fragments. The input assumption remains part of interpreting any later protein-encoder or fusion gain.


---

## Deviations recorded for Phase A

The audit identifies what the available comparison can preserve and where its relationship to the paper remains uncertain.

| Deviation or limitation | Evidence in this audit | Treatment in the current pipeline |
| --- | --- | --- |
| Ab1 is unavailable | No file was located in the inspected public release | Compare four benchmarks and state the omission; obtaining Ab1 remains separate work |
| The 999-nucleotide filter is inferred | Dropping longer rows matches retained counts; the paper describes truncation | Use `blf_benchmark_rows` consistently without claiming recovered historical row identity |
| Stability repeats sequences across supplied splits | About 68% of retained test rows reuse a training sequence | Retain the supplied split for comparison; sequence-grouped evaluation remains separate work |
| Some sequences lack a verified protein reading frame | Especially relevant to the designed CoV-Vac fragments | Step 2 translates from the supplied origin, replacing stops with `X`; this is a computational input assumption |

This audit checks dataset inputs, not the fusion architecture or training recipe. [Step 3](blf_build_check.ipynb) owns the TextCNN, projection and codon-grid implementation choices and their comparisons; the [Stage 4 runbook](../stage4/RUNBOOK.md) records the later evaluation and reuse procedures.


---

## What step 2 needs from here

The saved checks support a consistent input route for four available benchmarks: the shared length filter, the supplied row order and split assignments, and explicit limits on interpreting repeated sequences or translated fragments. They do not establish complete equivalence to the paper's historical dataset.

[Step 2, token embeddings](blf_token_embeddings.ipynb), uses:

- The rows returned by `blf_benchmark_rows`, in file order, so each cached token array stays paired with its target and split.
- [`token_embeddings.inputs_for_rows`](../shared_code/fusion/token_embeddings.py) to supply each encoder's alphabet and the grid-preserving protein translation.
- Encoder-specific token checks. For a 999-nucleotide A/C/G/U input, the helper predicts 999 RNA-FM content tokens, 333 protein-input tokens and 169 NT content tokens: 166 complete six-mers plus three single-base remainder tokens. Encoder inputs also include framing tokens, which are removed from the saved arrays. Extraction checks the actual input against the selected encoder's limit and rejects overlength inputs rather than silently truncating them.

Token-level embeddings retain positions that Stage 1's pooled vectors summarize away: an RNA-FM row can contain 999 content tokens with 640 features each. The caches therefore need substantially more storage than pooled vectors. Step 2 owns storage and runtime projections; its embedding files are local, ignored artifacts under the repository's [cache policy](../../README.md#results-caches-and-intermediate-files).
